# Módulo 5: Cálculo II
## Lección 10: Transformaciones de Coordenadas, Rotaciones y Tensores

Las **transformaciones de coordenadas** son herramientas matemáticas fundamentales en el análisis de sistemas físicos y geométricos. Permiten cambiar el marco de referencia espacial para describir de forma natural la cinemática de cuerpos rígidos, simplificar ecuaciones diferenciales complejas mediante simetrías axiales o radiales, o formular leyes de conservación covariantes independientes del sistema de coordenadas elegido.

En esta lección, estudiaremos formalmente las rotaciones en el plano bidimensional y en el espacio tridimensional. Analizaremos las rotaciones desde la perspectiva de la teoría de grupos, utilizando **generadores infinitesimales** y la **exponenciación de matrices** para construir transformaciones continuas. Posteriormente, definiremos los **tensores** y exploraremos su utilidad física mediante tensores de tensión, de inercia y electromagnéticos. Finalmente, deduciremos los **elementos de línea y de volumen** en coordenadas curvilíneas y aplicaremos transformaciones en coordenadas cartesianas, cilíndricas y esféricas.

### Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Construir y aplicar matrices de rotación en 2D y 3D** para transformar vectores y geometrías poligonales.
2. **Entender la relación exponencial** entre los generadores infinitesimales $J$ y el grupo de rotaciones $SO(2)$ mediante la serie de Taylor de matrices.
3. **Rotar vectores en 3D sobre cualquier eje arbitrario** aplicando la fórmula vectorial de Rodrigues.
4. **Definir e interpretar tensores de rango 2** en contextos físicos (tensiones en elásticos, momento de inercia en sólidos, tensor electromagnético en relatividad).
5. **Deducir los factores de escala y elementos diferenciales de línea $ds$ y de volumen $dV$** en coordenadas cilíndricas y esféricas.
6. **Convertir coordenadas de puntos y campos** entre sistemas cartesianos, cilíndricos y esféricos usando Python.

### 1. Rotaciones en el Plano (2D)

Una rotación en el plano por un ángulo $\theta$ alrededor del origen es una transformación lineal que preserva la longitud de los vectores y los ángulos entre ellos. Se representa mediante la **matriz de rotación** $R(\theta)$:

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

Si $\mathbf{p} = (x, y)$ es un punto en el plano cartesiano, su posición rotada $\mathbf{p}' = (x', y')$ se calcula mediante el producto matricial:

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix}$$

Esta transformación gira el punto en **sentido antihorario** para un ángulo $\theta > 0$.

Visualizaremos a continuación cómo se rotan los vértices de un triángulo en el plano por un ángulo ajustable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la
import sympy as sp

# Configuración de estilo visual
plt.style.use('seaborn-v0_8-whitegrid')

# Definición de vértices de un triángulo original: A(1,0), B(0,1), C(-1,0)
triangulo = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [-1.0, 0.0],
    [1.0, 0.0]  # Cerrar el triángulo para graficar
])

# Angulo de rotación de 30 grados (pi/6)
theta = np.pi / 6.0
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)]
])

# Rotar todos los vértices del triángulo
triangulo_rotado = np.dot(triangulo, R.T)

plt.figure(figsize=(6, 6))
plt.plot(triangulo[:,0], triangulo[:,1], 'b-o', linewidth=2.5, label='Triángulo Original')
plt.plot(triangulo_rotado[:,0], triangulo_rotado[:,1], 'r--s', linewidth=2.5, label=r'Triángulo Rotado ($30^\circ$)')
plt.axhline(0, color='black', linewidth=1, alpha=0.5)
plt.axvline(0, color='black', linewidth=1, alpha=0.5)
plt.title('Rotación de un Triángulo en el Plano 2D')
plt.xlabel('X')
plt.ylabel('Y')
plt.axis('equal')
plt.xlim(-1.5, 1.5)
plt.ylim(-1.5, 1.5)
plt.legend(loc='upper right', frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 2. Generadores del Grupo de Rotaciones (SO(2))

En la teoría de grupos de Lie, las rotaciones bidimensionales forman el grupo ortogonal especial $SO(2)$. Todas las matrices de este grupo pueden generarse exponencialmente a partir de un único **generador infinitesimal** $J$, que describe rotaciones infinitesimales cerca de la identidad:

$$J = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$$

La exponenciación de esta matriz se define mediante su **serie de Taylor**:

$$e^{\theta J} = \sum_{n=0}^{\infty} \frac{(\theta J)^n}{n!} = I + \theta J + \frac{\theta^2}{2!} J^2 + \frac{\theta^3}{3!} J^3 + \dots$$

Dado que las potencias de $J$ son periódicas ($J^2 = -I$, $J^3 = -J$, $J^4 = I$), la serie se divide en dos componentes:

$$e^{\theta J} = \left( 1 - \frac{\theta^2}{2!} + \frac{\theta^4}{4!} - \dots \right) I + \left( \theta - \frac{\theta^3}{3!} + \frac{\theta^5}{5!} - \dots \right) J = (\cos\theta) I + (\sin\theta) J = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} = R(\theta)$$

A continuación, implementaremos numéricamente la serie de Taylor para $e^{\theta J}$ evaluando su velocidad de convergencia en comparación con la función `scipy.linalg.expm`.

In [ ]:
def exp_matriz_taylor(A, theta, M):
    theta_A = theta * A
    suma = np.eye(A.shape[0])
    termino = np.eye(A.shape[0])
    for n in range(1, M + 1):
        termino = np.dot(termino, theta_A) / n
        suma += termino
    return suma

J = np.array([[0.0, -1.0], [1.0, 0.0]])
theta_val = np.pi / 4.0 # 45 grados
R_real = np.array([
    [np.cos(theta_val), -np.sin(theta_val)],
    [np.sin(theta_val), np.cos(theta_val)]
])

print("CONVERGENCIA DE LA SERIE DE TAYLOR EXPO-MATRICIAL:")
terminos = list(range(1, 12))
errores = []

for M in terminos:
    R_aprox = exp_matriz_taylor(J, theta_val, M)
    err = np.linalg.norm(R_aprox - R_real)
    errores.append(err)
    if M in [1, 3, 5, 8, 10]:
        print(f"  Términos Taylor (M={M:2d}) | Error Frobenius: {err:.2e}")

# Graficar convergencia del error
plt.figure(figsize=(7, 4.5))
plt.semilogy(terminos, errores, 'r-o', linewidth=2.0, label='Error Serie de Taylor')
plt.title('Convergencia del Error Exponencial Matricial')
plt.xlabel('Número de Términos (M)')
plt.ylabel('Error de Frobenius (Escala Log)')
plt.legend(frameon=True)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3. Rotaciones en 3D y Fórmula de Rodrigues

En tres dimensiones, las rotaciones forman el grupo de Lie $SO(3)$. Las rotaciones principales alrededor de los ejes coordenados $x$, $y$, $z$ se expresan como:

$$R_z(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{pmatrix}, \quad R_y(\theta) = \begin{pmatrix} \cos\theta & 0 & \sin\theta \\ 0 & 1 & 0 \\ -\sin\theta & 0 & \cos\theta \end{pmatrix}, \quad R_x(\theta) = \begin{pmatrix} 1 & 0 & 0 \\ 0 & \cos\theta & -\sin\theta \\ 0 & \sin\theta & \cos\theta \end{pmatrix}$$

#### Fórmula de Rotación de Rodrigues:
Si deseamos rotar un vector $\mathbf{v}$ en el espacio un ángulo $\theta$ alrededor de un **eje unitario arbitrario** $\mathbf{n}$ (donde $\|\mathbf{n}\| = 1$), se aplica directamente la fórmula vectorial de Rodrigues:

$$\mathbf{v}_{rot} = \mathbf{v} \cos\theta + (\mathbf{n} \times \mathbf{v}) \sin\theta + \mathbf{n} (\mathbf{n} \cdot \mathbf{v}) (1 - \cos\theta)$$

Esta fórmula calcula la rotación sin necesidad de construir matrices complejas de cambio de base.

In [ ]:
# Rotación de Rodrigues en 3D
def rotacion_rodrigues(v, n_axis, theta):
    n = n_axis / np.linalg.norm(n_axis)
    v_rot = v * np.cos(theta) + np.cross(n, v) * np.sin(theta) + n * np.dot(n, v) * (1.0 - np.cos(theta))
    return v_rot

v_orig = np.array([1.0, 0.0, 0.0])
eje_diag = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0) # diagonal unitaria
angulo_rot = np.pi / 4.0 # 45 grados

v_r = rotacion_rodrigues(v_orig, eje_diag, angulo_rot)
print("VERIFICACIÓN DE ROTACIÓN 3D DE RODRIGUES:")
print("  Vector original:          ", v_orig)
print("  Eje de Rotación (n):      ", eje_diag)
print("  Vector Rotado 45 grados:  ", v_r)
print(f"  Magnitud vector rotado:   {np.linalg.norm(v_r):.4f} (debe ser 1.0)")

### 4. Tensores y sus Aplicaciones en la Física

Los **tensores** son generalizaciones de los conceptos de escalares, vectores y matrices a dimensiones arbitrarias. Una de sus propiedades clave es la **covarianza**, lo que significa que sus leyes de transformación bajo cambios de coordenadas garantizan que las ecuaciones físicas sean idénticas en cualquier marco de referencia.

En mecánica clásica y relatividad, se utilizan tensores de rango 2 representados por matrices cuadradas:
- **Tensor de Tensiones de Cauchy ($\sigma_{ij}$):** Describe las fuerzas internas (esfuerzo cortante y tracción) por unidad de área dentro de un material deformable.
- **Tensor de Inercia ($I_{ij}$):** Caracteriza la distribución de masa y la resistencia a la aceleración angular de un cuerpo rígido sobre los tres ejes.
- **Tensor Electromagnético de Faraday ($F_{\mu\nu}$):** En relatividad especial, unifica las componentes de los campos eléctricos y magnéticos en una matriz totalmente antisimétrica de tamaño $4 \times 4$:
  
  $$F_{\mu\nu} = \begin{pmatrix} 0 & E_x & E_y & E_z \\ -E_x & 0 & B_z & -B_y \\ -E_y & -B_z & 0 & B_x \\ -E_z & B_y & -B_x & 0 \end{pmatrix}$$

Visualizaremos la distribución de este tensor electromagnético en un mapa de calor.

In [ ]:
# Mapa de calor del tensor electromagnético
# Daremos valores arbitrarios a los campos para visualización didáctica
Ex, Ey, Ez = 2.0, 1.5, -1.0
Bx, By, Bz = 0.5, -2.5, 3.0

F = np.array([
    [0.0, Ex, Ey, Ez],
    [-Ex, 0.0, Bz, -By],
    [-Ey, -Bz, 0.0, Bx],
    [-Ez, By, -Bx, 0.0]
])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(F, cmap='RdBu', interpolation='nearest')
cbar = fig.colorbar(im)
cbar.set_label('Intensidad de Componente')

ax.set_xticks(np.arange(4))
ax.set_yticks(np.arange(4))
ax.set_xticklabels([r'$\mu=0$', r'$\mu=1$', r'$\mu=2$', r'$\mu=3$'])
ax.set_yticklabels([r'$\nu=0$', r'$\nu=1$', r'$\nu=2$', r'$\nu=3$'])
ax.set_title('Mapa de Calor del Tensor Electromagnético $F_{\\mu\\nu}$')
plt.tight_layout()
plt.show()

### 5. Elementos Curvilíneos y Conversiones de Coordenadas

#### Elemento de Línea ($ds$):
En coordenadas curvilíneas, el elemento diferencial de línea se escribe en función de los **factores de escala** $h_i$ de cada coordenada:

$$ds^2 = h_1^2 du_1^2 + h_2^2 du_2^2 + h_3^2 du_3^2$$
- Cartesianas: $ds^2 = dx^2 + dy^2 + dz^2$ ($h_x=h_y=h_z=1$).
- Cilíndricas: $ds^2 = dr^2 + r^2 d\theta^2 + dz^2$ ($h_r=1, \ h_\theta=r, \ h_z=1$).
- Esféricas: $ds^2 = dr^2 + r^2 d\phi^2 + r^2 \sin^2\phi \, d\theta^2$ ($h_r=1, \ h_\phi=r, \ h_\theta=r\sin\phi$ en la convención física donde $\phi$ es el ángulo polar).

#### Elemento de Volumen ($dV$):
El diferencial de volumen se calcula multiplicando los factores de escala:

$$dV = h_1 h_2 h_3 \, du_1 \, du_2 \, du_3$$
- Cartesianas: $dV = dx \, dy \, dz$.
- Cilíndricas: $dV = r \, dr \, d\theta \, dz$.
- Esféricas: $dV = r^2 \sin\phi \, dr \, d\phi \, d\theta$.

#### Fórmulas de Conversión Esférica a Cartesiana:

$$x = r \sin\phi \cos\theta$$
$$y = r \sin\phi \sin\theta$$
$$z = r \cos\phi$$

Utilizaremos `SymPy` para calcular simbólicamente esta conversión para el punto esférico $P(r=2, \theta=\pi/6, \phi=\pi/4)$.

In [ ]:
r_sym, th_sym, phi_sym = sp.symbols('r theta phi', real=True)

# Ecuaciones de conversión
x_eq = r_sym * sp.sin(phi_sym) * sp.cos(th_sym)
y_eq = r_sym * sp.sin(phi_sym) * sp.sin(th_sym)
z_eq = r_sym * sp.cos(phi_sym)

# Evaluar en el punto P(2, pi/6, pi/4)
punto_esf = {r_sym: 2, th_sym: sp.pi/6, phi_sym: sp.pi/4}
x_exact = sp.simplify(x_eq.subs(punto_esf))
y_exact = sp.simplify(y_eq.subs(punto_esf))
z_exact = sp.simplify(z_eq.subs(punto_esf))

print("CONVERSIÓN SIMBÓLICA CON SYMPY (Esféricas -> Cartesianas):")
print(f"  Coordenada X exacta: {x_exact}  (Esperado: sqrt(6)/2)")
print(f"  Coordenada Y exacta: {y_exact}  (Esperado: sqrt(2)/2)")
print(f"  Coordenada Z exacta: {z_exact}  (Esperado: sqrt(2))")

assert x_exact == sp.sqrt(6)/2
assert y_exact == sp.sqrt(2)/2
assert z_exact == sp.sqrt(2)
print("\n¡Conversión simbólica verificada con éxito!")

### Resumen de Conceptos Clave

1. **Matrices de Rotación:** Operadores ortogonales que preservan longitudes y ángulos en $\mathbb{R}^2$ y $\mathbb{R}^3$.
2. **Generadores e Exponenciación:** La matriz de rotación plana $R(\theta)$ se construye como la exponencial de matriz $e^{\theta J}$, convergente a través de series de Taylor.
3. **Fórmula de Rodrigues:** Permite realizar rotaciones tridimensionales sobre cualquier eje $\mathbf{n}$ arbitrario sin construir cambios de base complejos.
4. **Tensores:** Estructuras covariantes invariantes bajo cambio de coordenadas, fundamentales para modelar tensiones elásticas, inercia angular y campos relativistas.
5. **Factores de Escala:** Definen la longitud diferencial $ds^2$ y el volumen $dV$ en coordenadas cilíndricas y esféricas.

### Referencias Bibliográficas

- Boyce, W. E., & DiPrima, R. C. (2012). *Elementary Differential Equations and Boundary Value Problems*. John Wiley & Sons.
- Marsden, J. E., & Tromba, A. J. (2012). *Cálculo Vectorial* (6ta ed.). Pearson.
- Boas, M. L. (2006). *Mathematical Methods in the Physical Sciences* (3rd ed.). Wiley.
- Feynman, R. P., Leighton, R. B., & Sands, M. (1964). *The Feynman Lectures on Physics: Vol. 2*. Addison-Wesley.